In [2]:
import pandas as pd
import glob
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import StackingRegressor
from xgboost import XGBRegressor
import math
import glob

In [3]:
files = glob.glob("41025h*.txt")

files


['41025h2014.txt',
 '41025h2015.txt',
 '41025h2016.txt',
 '41025h2017.txt',
 '41025h2018.txt',
 '41025h2019.txt']

In [4]:
na_vals = [99, 999, 9999, 99999]

def read_ndbc_stdmet(path):
    # 1) Read the header line that starts with #YY
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        header = None
        for line in f:
            if line.startswith("#YY"):
                header = line.lstrip("#").strip().split()
                break
    if header is None:
        raise ValueError(f"Couldn't find #YY header in {path}")

    # 2) Read the data, skipping comment lines, and apply the header we extracted
    df = pd.read_csv(
        path,
        sep=r"\s+",
        comment="#",
        header=None,
        names=header,
        na_values=na_vals,
        engine="python",
    )
    return df

files = sorted(glob.glob("41025h*.txt"))  # adjust if your files end with .txt

dfs = [read_ndbc_stdmet(f) for f in files]
wave_data = pd.concat(dfs, ignore_index=True)

print(wave_data.shape)
print(wave_data.columns.tolist())
wave_data.head()


(117882, 18)
['YY', 'MM', 'DD', 'hh', 'mm', 'WDIR', 'WSPD', 'GST', 'WVHT', 'DPD', 'APD', 'MWD', 'PRES', 'ATMP', 'WTMP', 'DEWP', 'VIS', 'TIDE']


,YY,MM,DD,hh,mm,WDIR,WSPD,GST,WVHT,DPD,APD,MWD,PRES,ATMP,WTMP,DEWP,VIS,TIDE
0,2013,12,31,23,50,305.0,7.0,10.0,0.96,5.56,4.41,2.0,1025.5,11.8,23.3,1.8,NaN,NaN
1,2014,1,1,0,50,301.0,6.8,9.5,0.96,6.25,4.40,360.0,1026.6,11.8,23.3,1.4,NaN,NaN
2,2014,1,1,1,50,305.0,7.5,9.5,0.89,5.88,4.23,2.0,1027.6,11.9,23.3,0.5,NaN,NaN
3,2014,1,1,2,50,308.0,6.6,9.4,0.89,5.88,4.15,3.0,1027.9,11.7,23.3,-0.1,NaN,NaN
4,2014,1,1,3,50,322.0,7.0,10.1,0.92,6.25,4.38,26.0,1027.9,12.0,23.2,-0.3,NaN,NaN


In [5]:
wave_data["datetime"] = pd.to_datetime(
    wave_data.rename(columns={"YY":"year","MM":"month","DD":"day","hh":"hour","mm":"minute"})[
        ["year","month","day","hour","minute"]
    ],
    errors="coerce"
)

wave_data = wave_data.dropna(subset=["datetime"]).set_index("datetime").sort_index()


In [6]:
wave_data.isna().sum().sort_values(ascending=False)

TIDE    117882
VIS     117882
MWD      73961
WVHT     73607
APD      73607
DPD      73607
DEWP     17540
ATMP      8042
WTMP      5272
WDIR      4505
GST       1089
WSPD      1086
PRES       212
MM           0
mm           0
hh           0
DD           0
YY           0
dtype: int64